# Official FactorizePhys+FSAM (SCAMPS pretrained) → UBFC Zero-Shot Eval

Loads `SCAMPS_FactorizePhys_FSAM_Res.pth` from the official FactorizePhys repo  
and evaluates it on UBFC-rPPG DATASET_2 using the same YOLO5Face pipeline as Exp 3.

**Purpose:** establish what the official pretrained model scores on our exact UBFC data,  
as a direct comparison to our Exp 3 result (1.70 bpm).

In [1]:
import sys, time, warnings
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import butter, filtfilt
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import imageio
import cv2

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('..').resolve()
FP_ROOT      = PROJECT_ROOT / 'external' / 'FactorizePhys'
WEIGHTS_PATH = FP_ROOT / 'final_model_release' / 'SCAMPS_FactorizePhys_FSAM_Res.pth'
UBFC_ROOT    = PROJECT_ROOT / 'rppg_dataset' / 'UBFC' / 'DATASET_2'
UBFC_CACHE   = Path('/tmp/ubfc_y5f_clips_72.pt')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if str(FP_ROOT) not in sys.path:
    sys.path.insert(0, str(FP_ROOT))

print('Device      :', device)
print('Weights     :', WEIGHTS_PATH.exists())
print('UBFC root   :', UBFC_ROOT.exists())
print('UBFC cache  :', UBFC_CACHE.exists())

Device      : cuda:0
Weights     : True
UBFC root   : True
UBFC cache  : True


In [2]:
# ── Load official FactorizePhys model + SCAMPS pretrained weights ─────────────
from yacs.config import CfgNode as CN
from neural_methods.model.FactorizePhys.FactorizePhys import FactorizePhys

md_cfg = CN()
md_cfg.CHANNELS     = 3
md_cfg.FRAME_NUM    = 160
md_cfg.MD_FSAM      = True
md_cfg.MD_TYPE      = 'NMF'
md_cfg.MD_R         = 1
md_cfg.MD_S         = 1
md_cfg.MD_STEPS     = 4
md_cfg.MD_RESIDUAL  = True
md_cfg.MD_INFERENCE = True

model = FactorizePhys(frames=160, md_config=md_cfg, device=device, in_channels=3).to(device)

ckpt      = torch.load(str(WEIGHTS_PATH), map_location='cpu', weights_only=False)
new_state = OrderedDict((k.replace('module.', ''), v) for k, v in ckpt.items())
missing, unexpected = model.load_state_dict(new_state, strict=False)

print(f'Params    : {sum(p.numel() for p in model.parameters()):,}')
print(f'Missing   : {missing}')       # expect ["rppg_head.bias1"] — scalar init to 1.0
print(f'Unexpected: {unexpected}')

model.eval()
print('Model ready.')

Params    : 52,168
Missing   : []
Unexpected: []
Model ready.


In [3]:
# ── UBFC Dataset (reuses /tmp/ubfc_y5f_clips_72.pt cache from notebook 03) ───
import types as _types
for _n in ['dataset', 'dataset.data_loader']:
    if _n not in sys.modules:
        _m = _types.ModuleType(_n)
        _m.__path__ = [str(FP_ROOT / _n.replace('.', '/'))]
        _m.__package__ = _n
        sys.modules[_n] = _m
from dataset.data_loader.face_detector.YOLO5Face import YOLO5Face as _YOLO5Face

_Y5F = None
def _get_y5f():
    global _Y5F
    if _Y5F is None:
        _Y5F = _YOLO5Face(backend='Y5F', device='cpu')
    return _Y5F

def _y5f_bbox(frame_rgb, img_h, img_w, coef=1.5):
    res = _get_y5f().detect_face(frame_rgb.astype(np.uint8))
    if res is None:
        return [0, 0, img_w, img_h]
    x1, y1, x2, y2 = res
    w, h = x2 - x1, y2 - y1
    sq   = max(w, h)
    cx, cy = x1 + w // 2, y1 + h // 2
    ex = max(0, cx - sq // 2 - (coef - 1.0) / 2 * sq)
    ey = max(0, cy - sq // 2 - (coef - 1.0) / 2 * sq)
    return [int(ex), int(ey), int(coef * sq), int(coef * sq)]

def _load_ubfc_y5f(vid_path, img_size=72, detect_every=30):
    raw_frames = []
    reader = imageio.get_reader(str(vid_path), format='pyav')
    for frame in reader: raw_frames.append(frame)
    reader.close()
    img_h, img_w = raw_frames[0].shape[:2]
    bboxes = {i: _y5f_bbox(raw_frames[i], img_h, img_w)
              for i in range(0, len(raw_frames), detect_every)}
    out = np.empty((len(raw_frames), img_size, img_size, 3), dtype=np.uint8)
    for i, frame in enumerate(raw_frames):
        x, y, w, h = bboxes[(i // detect_every) * detect_every]
        crop = frame[max(y,0):min(y+h,img_h), max(x,0):min(x+w,img_w)]
        out[i] = cv2.resize(crop, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return out


class UBFCDataset(Dataset):
    def __init__(self, ubfc_root, clip_len=161, img_size=72):
        self.clip_len = clip_len
        if UBFC_CACHE.exists():
            print(f'Loading UBFC cache: {UBFC_CACHE}')
            self.clips = torch.load(str(UBFC_CACHE), weights_only=False)
            print(f'  {len(self.clips)} clips ready.')
            return
        subject_dirs = sorted(Path(ubfc_root).glob('subject*'),
                              key=lambda p: int(p.name.replace('subject', '')))
        print(f'YOLO5Face preprocessing {len(subject_dirs)} subjects...')
        self.clips = []
        chunk_len = clip_len - 1
        for i, sdir in enumerate(subject_dirs):
            frames_np = _load_ubfc_y5f(sdir / 'vid.avi', img_size).astype(np.float32) / 255.0
            frames_t  = torch.from_numpy(np.ascontiguousarray(frames_np.transpose(3,0,1,2)))
            ppg_full  = np.loadtxt(str(sdir / 'ground_truth.txt'))[0].astype(np.float32)
            T = frames_t.shape[1]
            for k in range(T // chunk_len):
                s = k * chunk_len
                f = frames_t[:, s:s+chunk_len]
                f = torch.cat([f, f[:, -1:]], dim=1)
                p = ppg_full[s:s+chunk_len].copy()
                std = p.std()
                if std > 1e-8: p = (p - p.mean()) / std
                self.clips.append({'frames': f, 'ppg': torch.from_numpy(p), 'subj': sdir.name})
            if (i+1) % 10 == 0: print(f'  {i+1}/{len(subject_dirs)}')
        torch.save(self.clips, str(UBFC_CACHE))
        print(f'{len(self.clips)} clips saved to {UBFC_CACHE}')

    def __len__(self): return len(self.clips)
    def __getitem__(self, idx): return self.clips[idx]


ubfc_ds     = UBFCDataset(UBFC_ROOT)
ubfc_loader = DataLoader(ubfc_ds, batch_size=8, shuffle=False,
                         num_workers=2, pin_memory=True)

Loading UBFC cache: /tmp/ubfc_y5f_clips_72.pt


  483 clips ready.


In [4]:
# ── Inference + HR extraction ─────────────────────────────────────────────────
def extract_hr(waveform, fps=30.0, low_bpm=36, high_bpm=198):
    lo, hi = low_bpm / 60.0, high_bpm / 60.0
    nyq = fps / 2.0
    N = len(waveform)
    if N < 8: return 75.0
    try:
        b, a = butter(4, [lo/nyq, hi/nyq], btype='bandpass')
        sig  = filtfilt(b, a, waveform.astype('float64'))
    except Exception:
        sig = waveform.astype('float64')
    fft  = np.abs(np.fft.rfft(sig * np.hanning(N)))
    freq = np.fft.rfftfreq(N, d=1.0/fps)
    mask = (freq >= lo) & (freq <= hi)
    return float(freq[mask][np.argmax(fft[mask])] * 60.0) if mask.any() else 75.0


pred_hrs, gt_hrs, pearsons = [], [], []
t0 = time.time()

with torch.no_grad():
    for batch in ubfc_loader:
        frames = batch['frames'].to(device)
        ppg_gt = batch['ppg'].numpy()
        out    = model(frames)
        preds  = out[0].float().cpu().numpy()
        for p, g in zip(preds, ppg_gt):
            pred_hrs.append(extract_hr(p))
            gt_hrs.append(extract_hr(g))
            if p.std() > 1e-8 and g.std() > 1e-8:
                pearsons.append(float(np.corrcoef(p, g)[0, 1]))

print(f'Inference done in {time.time()-t0:.1f}s')

pred_hrs = np.array(pred_hrs)
gt_hrs   = np.array(gt_hrs)
err      = pred_hrs - gt_hrs
mae      = float(np.abs(err).mean())
rmse     = float(np.sqrt((err**2).mean()))
mape     = float((np.abs(err) / (np.abs(gt_hrs) + 1e-8)).mean() * 100)
r        = float(np.mean(pearsons))

sep = '=' * 65
print(sep)
print('Official SCAMPS_FactorizePhys_FSAM_Res.pth → UBFC zero-shot')
print(sep)
print(f'  HR MAE    : {mae:.2f} bpm   (paper reports: 1.17 bpm)')
print(f'  HR RMSE   : {rmse:.2f} bpm')
print(f'  HR MAPE   : {mape:.2f} %')
print(f'  Pearson r : {r:.4f}')
print(f'  N clips   : {len(pred_hrs)}')
print(sep)
print()
print('  Our Exp 3  (1 epoch, same protocol): MAE=1.70 bpm, r=0.514')
print(f'  This model (official weights)      : MAE={mae:.2f} bpm, r={r:.4f}')
print(sep)

Inference done in 4.3s
Official SCAMPS_FactorizePhys_FSAM_Res.pth → UBFC zero-shot
  HR MAE    : 1.49 bpm   (paper reports: 1.17 bpm)
  HR RMSE   : 6.47 bpm
  HR MAPE   : 1.60 %
  Pearson r : 0.5435
  N clips   : 483

  Our Exp 3  (1 epoch, same protocol): MAE=1.70 bpm, r=0.514
  This model (official weights)      : MAE=1.49 bpm, r=0.5435


In [5]:
# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(gt_hrs, pred_hrs, alpha=0.4, s=12, color='steelblue')
lims = [36, 198]
axes[0].plot(lims, lims, 'r--', lw=1)
axes[0].set_xlabel('Ground Truth HR (bpm)')
axes[0].set_ylabel('Predicted HR (bpm)')
axes[0].set_title(f'Official weights → UBFC | MAE={mae:.2f} bpm')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)

axes[1].hist(err, bins=30, edgecolor='black', color='steelblue')
axes[1].axvline(0, color='red', ls='--', lw=1)
axes[1].set_xlabel('HR Error (bpm)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Error distribution | RMSE={rmse:.2f} bpm')

plt.tight_layout()
out_path = PROJECT_ROOT / 'docs' / 'official_factorizephys_ubfc_eval.png'
plt.savefig(str(out_path), dpi=150)
plt.show()
print(f'Plot saved → {out_path}')

Plot saved → /mnt/sata-ssd/rppg_project/docs/official_factorizephys_ubfc_eval.png
